# Agentic Components: Exhaustive Feature Showcase

A guided, multi-step notebook that demonstrates the full framework: prompts, output parsing, memory, tools, comms, rate limits, logging/tracing, async/parallel orchestration, dynamic graphs, vector retrieval, and a complex release-readiness use case.

In [ ]:
# Runtime setup: optional OpenAI key, fallback to stub LLM
import os, re, json
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter OPENAI_API_KEY (press Enter to skip): ')

from agentic_codex import (
    AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter,
    CommunicationHub, CommEnvelope, PromptManager, Prompt_Framework,
    MultiLimiter, RateLimiter, SafeFileReaderToolAdapter, SanitizedCodeExecutionToolAdapter,
    HashEmbeddingAdapter, VectorMemory, RunStore, StructuredLogger,
    StepSpec, run_steps, GraphRunner, GraphNodeSpec, apply_rate_limits, load_prompts,
    SQLiteMemory, parse_llm_json, validate_with_model, build_output
)
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.observability.tracer import Tracer
from agentic_codex.core.observability.prometheus import render_metrics
from pydantic import BaseModel

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return '[stub llm] ' + (lines[-1][:200] if lines else 'response')

try:
    llm = EnvOpenAIAdapter(model='gpt-4o-mini')
except Exception:
    llm = FunctionAdapter(stub_llm)


## Prompt management & frameworks
- Register versioned prompts with PromptManager.
- Generate framework-specific prompts via Prompt_Framework.

In [ ]:
prompt_manager = PromptManager()
prompt_manager.add('review_prompt', 'Review code and score 0-10; respond JSON.', version='v1')
prompt_manager.add('doc_prompt', 'Document the changes and risks.', version='v1')
review_prompt = prompt_manager.get('review_prompt')
doc_prompt = prompt_manager.get('doc_prompt')

pf = Prompt_Framework(context='Refactor calculator for reliability', output_type='code review', style='formal')
pf.switch_framework('care')
care_prompt = pf.generate_prompt(action='refactor', result='clean code', example='use functions')
review_prompt, doc_prompt, care_prompt[:160]


## Output parsing & formatting
- Enforce `<json>...</json>` in LLM output, regex extract, validate with Pydantic, render in multiple formats.

In [ ]:
class ReviewSchema(BaseModel):
    score: float
    issues: list[str]

llm_output = '<json>{"score": 9.6, "issues": ["spacing", "docstrings"]}</json>'
m = re.search(r'<json>(.*?)</json>', llm_output, re.DOTALL)
json_block = m.group(1) if m else '{}'
parsed = parse_llm_json(json_block)
validated = validate_with_model(ReviewSchema, parsed)
json_out = build_output(validated.model_dump(), fmt='json')
xml_out = build_output(validated.model_dump(), fmt='xml', root_tag='review')
html_out = build_output(validated.model_dump(), fmt='html', title='Review')
json_out[:120], xml_out[:120], html_out[:120]


## Memory, vector search, and safe tools
- SQLiteMemory for key/value.
- VectorMemory with hash embeddings for similarity search.
- Safe file read + sanitized code exec adapters.

In [ ]:
sql_mem = SQLiteMemory(':memory:')
sql_mem.put('cfg', 'release=true')
sql_hits = sql_mem.search('release')

vec_mem = VectorMemory(HashEmbeddingAdapter(dimension=32).embed)
vec_mem.add('reqs', 'Calculator must support CLI and logging')
vec_hits = vec_mem.search('logging', k=1)

safe_reader = SafeFileReaderToolAdapter(root='.')
safe_exec = SanitizedCodeExecutionToolAdapter()
sql_hits, vec_hits


## Communication, namespaces, rate limits
- Namespaces with allow/deny via CommunicationHub; private doc messages and broadcast.
- Rate limits per namespace/tool via MultiLimiter/CommunicationHub.

In [ ]:
hub = CommunicationHub()
hub.create_namespace('team', ['coder','reviewer','recommender'])
hub.create_namespace('docs', ['doc'])
hub.link_namespaces('team', 'docs')
hub.set_namespace_rate_limit('team', rate=5, capacity=5)

hub.send(CommEnvelope(sender='doc', mode='namespace', targets=['docs'], content='doc update 1', meta={'namespace':'docs'}))
hub.send(CommEnvelope(sender='loop', mode='broadcast', targets=[], content='Goal reached: score >= 9.5'))
[(rec.agent, rec.channel, rec.content) for rec in hub.conversation()]


## Observability: logging, tracing, metrics
- StructuredLogger for JSON logs.
- Tracer spans/metrics; Prometheus text rendering.

In [ ]:
logger = StructuredLogger(name='agentic-demo')
tracer = Tracer(run_id='obs-demo')
with tracer.span('demo-span', step='example'):
    tracer.metric('demo.metric', 1, tag='x')
logger.log('demo.log', detail='span complete')
metrics_data = {'counters': {('demo.metric', (('tag','x'),)): 1}, 'latency': {}}
prom_text = render_metrics(metrics_data)
prom_text


## Parallel/async batch steps (with_steps)
- Async worker runs over batch items in parallel threads.

In [ ]:
import asyncio
from agentic_codex import StepSpec, run_steps

async def async_worker(ctx: Context) -> AgentStep:
    await asyncio.sleep(0)
    item = ctx.scratch.get('batch_item')
    return AgentStep(out_messages=[Message(role='async_worker', content=f'async:{item}')], state_updates={'processed': item})

ctx_parallel = Context(goal='parallel-async-demo', scratch={'items': [1,2,3]})
specs = [StepSpec(name='async_batch', fn=async_worker, batch_key='items', parallel=True)]
parallel_results = run_steps(specs, ctx_parallel, max_parallel=3)
[m.content for r in parallel_results for m in r.out_messages]


## Dynamic graph mutation (GraphRunner)
- Add new nodes at runtime via `_graph_additions`.

In [ ]:
def base_step(ctx: Context) -> AgentStep:
    ctx.scratch.setdefault('_graph_additions', []).append(('extra', GraphNodeSpec(agent=extra_agent)))
    return AgentStep(out_messages=[Message(role='base', content='base run')])

def extra_step(ctx: Context) -> AgentStep:
    return AgentStep(out_messages=[Message(role='extra', content='extra run')])

base_agent = AgentBuilder(name='base', role='base').with_llm(llm).with_step(base_step).build()
extra_agent = AgentBuilder(name='extra', role='extra').with_llm(llm).with_step(extra_step).build()
graph = {'base': GraphNodeSpec(agent=base_agent)}
runner = GraphRunner(graph, branch_budget=1)
ctx_dyn = Context(goal='dyn graph', scratch={})
dyn_result = runner.run(goal=ctx_dyn.goal, inputs=ctx_dyn.scratch)
[(m.role, m.content) for m in dyn_result.messages]


## Complex use case: Release readiness loop
- Agents: coder, reviewer, recommender, doc.
- Uses prompt templates, comms (private doc updates, broadcast on success), logging, rate limits, vector hints.
- Loop runs until score ≥ 9.5 or iteration cap.

In [ ]:
logger.set_run_id('release-run')

def coder_step(ctx: Context) -> AgentStep:
    reqs = vec_mem.search('calculator')
    req_text = reqs[0][0] if reqs else 'Build calculator'
    prompt = f"Build/Improve calculator code. Requirements: {req_text}. Current code:\n{ctx.scratch.get('code','')}"
    code = ctx.llm.generate(prompt)
    hub.send(CommEnvelope(sender='coder', mode='namespace', targets=['docs'], content='coder updated code', meta={'namespace':'docs'}))
    logger.log('agent.coder', detail='updated code')
    return AgentStep(out_messages=[Message(role='coder', content=code)], state_updates={'code': code})

def reviewer_step(ctx: Context) -> AgentStep:
    prompt = review_prompt + f"\nCode:\n{ctx.scratch.get('code','')}\nWrap JSON in <json> tags."
    raw = ctx.llm.generate(prompt)
    m = re.search(r'<json>(.*?)</json>', raw, re.DOTALL)
    block = m.group(1) if m else raw
    parsed = parse_llm_json(block)
    score = float(parsed.get('score', 0)) if isinstance(parsed, dict) else 0.0
    hub.send(CommEnvelope(sender='reviewer', mode='namespace', targets=['docs'], content='review complete', meta={'namespace':'docs'}))
    logger.log('agent.reviewer', detail='scored code', score=score)
    return AgentStep(out_messages=[Message(role='reviewer', content=str(parsed))], state_updates={'score': score, 'review': parsed})

def recommender_step(ctx: Context) -> AgentStep:
    prompt = f"Suggest concrete fixes to reach 10/10. Review: {ctx.scratch.get('review','')}"
    recs = ctx.llm.generate(prompt)
    hub.send(CommEnvelope(sender='recommender', mode='namespace', targets=['docs'], content='recs ready', meta={'namespace':'docs'}))
    logger.log('agent.recommender', detail='provided recs')
    return AgentStep(out_messages=[Message(role='recommender', content=recs)], state_updates={'recs': recs})

def doc_step(ctx: Context) -> AgentStep:
    prompt = doc_prompt + f"\nCode excerpt:\n{ctx.scratch.get('code','')[:200]}\nScore: {ctx.scratch.get('score',0)}\nRecs: {ctx.scratch.get('recs','')}"
    doc = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='doc', content=doc)], state_updates={'docs': doc})

coder = AgentBuilder(name='coder', role='dev').with_llm(llm).with_step(coder_step).build()
reviewer = AgentBuilder(name='reviewer', role='qa').with_llm(llm).with_step(reviewer_step).build()
recommender = AgentBuilder(name='recommender', role='advisor').with_llm(llm).with_step(recommender_step).build()
doc_agent = AgentBuilder(name='doc', role='documentation').with_llm(llm).with_step(doc_step).build()

ctx = Context(goal='Ship calculator release', scratch={'score': 0.0}, components={'logger': logger})
iteration = 0
while ctx.scratch.get('score', 0.0) < 9.5 and iteration < 8:
    for agent in (coder, reviewer, recommender):
        result = agent.run(ctx)
        ctx.scratch.update(result.state_updates)
        doc_res = doc_agent.run(ctx)
        ctx.scratch.update(doc_res.state_updates)
    iteration += 1

hub.send(CommEnvelope(sender='loop', mode='broadcast', targets=[], content='Goal reached: score >= 9.5'))
store = RunStore('runs/')
store.save({'run_id':'release-run', 'score': ctx.scratch.get('score',0.0)})
ctx.scratch.get('score', 0.0), ctx.scratch.get('docs','')[:300]


## Cancel tokens for cooperative stop
Use `CancelToken` to halt orchestration early (with_steps/GraphRunner honor it).

In [ ]:
from agentic_codex import CancelToken
token = CancelToken()
ctx_cancel = Context(goal='cancel-demo', scratch={}, components={'cancel_token': token})
token.cancel('no more work')
res = coder.run(ctx_cancel)
res.stop, ctx_cancel.scratch.get('reason')


## Plugin discovery
Load adapters registered via entry points (`agentic_codex.*_adapters`).

In [ ]:
from agentic_codex import load_plugins
plugins = load_plugins()
plugins


## Manifest-style rate limits/prompts application
Demonstrates applying manifest `rate_limits` and `prompts` blocks programmatically.

In [ ]:
from agentic_codex import apply_rate_limits, load_prompts
from agentic_codex.manifests.validators import WorkflowManifest
manifest_like = WorkflowManifest(
    name='demo',
    steps=[],
    policies={},
    rate_limits={
        'namespaces': {'team': {'rate': 5, 'capacity': 5}},
        'tools': {'fs.safe_read': {'rate': 2, 'capacity': 2}},
    },
    prompts={'demo': {'template': 'Hello', 'version': 'v1'}},
)
apply_rate_limits(manifest_like, limiter=limiter, hub=hub)
load_prompts(manifest_like, prompt_manager)
prompt_manager.get('demo')


## HTTP service skeleton (FastAPI)
Build the service app; exposes /healthz, /run, /metrics, /jobs/{id}.

In [ ]:
from agentic_codex.service.http import build_app
app = build_app(run_store='runs/')
app.title, [route.path for route in app.routes][:5]


## Wrap-up
You now have examples covering prompt/versioning, strict output parsing, memory (SQL + vector), safe tools, comms, rate limits, logging/tracing/metrics, async/parallel steps, dynamic graphs, and a complex use case with documentation and broadcast signaling.